# 17.2 Domain adaptation and domain generalization

Learning must keep the label rule while refusing to memorize the domain a sample came from.

Source risk alone does not certify target risk. Mean alignment is useful when domain shift is nuisance, and dangerous when the shifted direction carries label information.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

The lesson diagnostic is the squared mean discrepancy $\|\mu_S-\mu_T\|_2^2$. Aligning target features by subtracting the source-to-target shift should take the worked discrepancy from 1.250 to 0.000.

In [ ]:

def domain_discrepancy(mu_s, mu_t):
    return float(np.sum((mu_s - mu_t) ** 2))

def mean_align(Xs, Xt):
    shift = Xt.mean(axis=0) - Xs.mean(axis=0)
    return Xt - shift, shift

mu_s = np.array([0.0, 0.0])
mu_t = np.array([1.0, -0.5])
before = domain_discrepancy(mu_s, mu_t)
aligned_mu_t = mu_t - (mu_t - mu_s)
after = domain_discrepancy(mu_s, aligned_mu_t)

assert round(before, 3) == 1.250
assert round(after, 3) == 0.000
print(f"before={before:.3f}, after={after:.3f}")


The same operation is wrapped in a classifier: fit on source, optionally mean-align target, then score target accuracy and discrepancy.

In [ ]:

def make_domain_ladder():
    digits = load_digits()
    X = digits.data / 16.0
    y = (digits.target >= 5).astype(int)
    rng = np.random.default_rng(172)
    specs = [
        ("D1 no synthetic shift", 0.0, False),
        ("D2 nuisance shift 0.25", 0.25, False),
        ("D3 partly label-correlated 0.50", 0.50, True),
        ("D4 iris-style offset 0.75", 0.75, False),
        ("D5 digits style shift 1.00", 1.00, True),
    ]
    rungs = []
    for name, mag, correlated in specs:
        source_idx, target_idx = train_test_split(np.arange(len(y)), test_size=0.45, random_state=int(mag * 100) + 4, stratify=y)
        Xs = X[source_idx].copy()
        ys = y[source_idx]
        Xt = X[target_idx].copy()
        yt = y[target_idx]
        direction = rng.normal(0.0, 1.0, size=X.shape[1])
        direction = direction / np.linalg.norm(direction)
        if correlated:
            Xt = Xt + mag * direction * (2 * yt[:, None] - 1)
        else:
            Xt = Xt + mag * direction
        rungs.append((name, Xs, ys, Xt, yt, mag, correlated))
    return rungs

def mean_align_then_fit(Xs, ys, Xt, yt, adapt=True, class_conditional=False):
    scaler = StandardScaler()
    Xs_scaled = scaler.fit_transform(Xs)
    Xt_scaled = scaler.transform(Xt)
    if adapt and class_conditional:
        pseudo = LogisticRegression(max_iter=1500).fit(Xs_scaled, ys).predict(Xt_scaled)
        Xt_fixed = Xt_scaled.copy()
        for cls in np.unique(ys):
            src_mean = Xs_scaled[ys == cls].mean(axis=0)
            tgt_mask = pseudo == cls
            if np.any(tgt_mask):
                tgt_mean = Xt_scaled[tgt_mask].mean(axis=0)
                Xt_fixed[tgt_mask] = Xt_scaled[tgt_mask] - (tgt_mean - src_mean)
        Xt_scaled = Xt_fixed
    elif adapt:
        Xt_scaled, _ = mean_align(Xs_scaled, Xt_scaled)
    discrepancy = domain_discrepancy(Xs_scaled.mean(axis=0), Xt_scaled.mean(axis=0))
    clf = LogisticRegression(max_iter=1500, C=2.0)
    clf.fit(Xs_scaled, ys)
    acc = accuracy_score(yt, clf.predict(Xt_scaled))
    return acc, discrepancy


## The dataset ladder

The rungs vary domain-shift magnitude on a real digits base. D3 and D5 deliberately make part of the shift label-correlated to test the lesson warning.

In [ ]:

domain_rungs = make_domain_ladder()

for name, Xs, ys, Xt, yt, mag, correlated in domain_rungs:
    print(f"{name:32s} source={Xs.shape} target={Xt.shape} shift={mag:.2f} correlated={correlated}")


In [ ]:

domain_results = []

for name, Xs, ys, Xt, yt, mag, correlated in domain_rungs:
    no_acc, no_disc = mean_align_then_fit(Xs, ys, Xt, yt, adapt=False)
    align_acc, align_disc = mean_align_then_fit(Xs, ys, Xt, yt, adapt=True)
    domain_results.append((name, mag, correlated, no_acc, align_acc, no_disc, align_disc, Xs, ys, Xt, yt))
    print(f"{name:32s} no={no_acc:.3f} align={align_acc:.3f} disc {no_disc:.3f}->{align_disc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], domain_results):
    name, mag, correlated, no_acc, align_acc, no_disc, align_disc, Xs, ys, Xt, yt = row
    both_X = np.vstack([Xs[:120], Xt[:120]])
    both_y = np.concatenate([np.zeros(120), np.ones(120)])
    plot_panel(ax, both_X, both_y, name.split()[0] + f" d={align_disc:.1f}")

shifts = [row[1] for row in domain_results]
no_scores = [row[3] for row in domain_results]
align_scores = [row[4] for row in domain_results]
axes[1, 0].plot(shifts, no_scores, marker="o", label="no adaptation")
axes[1, 0].plot(shifts, align_scores, marker="s", label="mean alignment")
axes[1, 0].set_xlabel("domain shift magnitude")
axes[1, 0].set_ylabel("target accuracy")
axes[1, 0].set_title("accuracy vs shift")
axes[1, 0].legend()

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: alignment removes label signal

Global alignment is unsafe when target shift is class-dependent. Class-conditional alignment gates the correction by pseudo-label and preserves more label information.

In [ ]:

name, Xs, ys, Xt, yt, mag, correlated = domain_rungs[-1]
no_acc, _ = mean_align_then_fit(Xs, ys, Xt, yt, adapt=False)
global_acc, _ = mean_align_then_fit(Xs, ys, Xt, yt, adapt=True)
class_acc, _ = mean_align_then_fit(Xs, ys, Xt, yt, adapt=True, class_conditional=True)

print(f"D5 no adaptation: {no_acc:.3f}")
print(f"D5 global mean alignment: {global_acc:.3f}")
print(f"D5 class-conditional alignment: {class_acc:.3f}")
assert class_acc >= global_acc - 0.05


## Evaluate it + Practice

- Metric: target accuracy vs domain-shift magnitude, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.